# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5.4-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [19]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system","content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type":"json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [20]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company overview', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum / offerings',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'program / product page',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project / product page',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project / product page',
   'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'posts / resources', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'social profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [21]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [22]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5.4-nano
Found 11 relevant links


{'links': [{'type': 'company overview', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company/programs', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'company/programs', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'featured project',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'featured project', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'news/blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'social profile (professional)',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social profile (updates)',
   'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social profile (updates)',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'external company/service',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [23]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5.4-nano
Found 20 relevant links


{'links': [{'type': 'home', 'url': 'https://huggingface.co/'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'docs', 'url': 'https://huggingface.co/docs'},
  {'type': 'changelog', 'url': 'https://huggingface.co/changelog'},
  {'type': 'learn', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'community discussion', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status', 'url': 'https://status.huggingface.co/'},
  {'type': 'developer portal / inference models',
   'url': 'https://huggingface.co/inference/models'},
  {'type': 'inference endpoints', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'careers', 

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [27]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [29]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5.4-nano
Found 32 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
SulphurAI/Sulphur-2-base
Updated
about 19 hours ago
•
115k
•
481
deepseek-ai/DeepSeek-V4-Pro
Updated
4 days ago
•
1.17M
•
3.78k
Zyphra/ZAYA1-8B
Updated
about 20 hours ago
•
23.6k
•
319
SeeSee21/Z-Anime
Updated
13 days ago
•
8.43k
•
260
TenStrip/LTX2.3-10Eros
Updated
3 days ago
•
51.8k
•
182
Browse 2M+ models
Spaces
Running
on
Zero
MCP
360
Qwen Image Edit + Loras built-in
👀
360
Qwen image edit with 🔞 loras
Running
on
Zero
MCP
1.04k
Wan2.2 14B Fast Preview
🐌
1.04k
generate a video from an image with a text prompt
Running
102
The ul

In [30]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [31]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [32]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5.4-nano
Found 16 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nSulphurAI/Sulphur-2-base\nUpdated\nabout 19 hours ago\n•\n115k\n•\n481\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n4 days ago\n•\n1.17M\n•\n3.78k\nZyphra/ZAYA1-8B\nUpdated\nabout 20 hours ago\n•\n23.6k\n•\n319\nSeeSee21/Z-Anime\nUpdated\n13 days ago\n•\n8.43k\n•\n260\nTenStrip/LTX2.3-10Eros\nUpdated\n3 days ago\n•\n51.8k\n•\n182\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n360\nQwen Im

In [33]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [34]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5.4-nano
Found 18 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the leading AI community dedicated to building the future of machine learning. As a vibrant platform, it fosters collaboration among passionate machine learning practitioners, researchers, and developers worldwide by providing access to an extensive ecosystem of models, datasets, and applications.

With a mission to democratize AI, Hugging Face empowers its users to **create, discover, and collaborate** on ML projects, enabling faster innovation through open-source tools and an inclusive, global community.

---

## Platform Highlights

### Massive Repository of Models  
- Browse over **2.8 million models** spanning diverse AI tasks such as text generation, image-to-text, text-to-video, and more.
- Models cater to various frameworks including PyTorch, TensorFlow, JAX, and specialized libraries like Transformers, Diffusers, and Sentence-Transformers.
- Cutting-edge models by leading AI developers and organizations, including ultra-large parameter models exceeding hundreds of billions.

### Extensive Datasets  
- Access nearly **1 million datasets** encompassing multiple data modalities: text, image, audio, video, 3D, geospatial, and tabular data.
- Datasets are optimized in various formats and cover use cases for benchmarking, training, and evaluation.
- Users benefit from constant updates of trending datasets to stay at the forefront of ML research.

### Innovative Applications & Spaces  
- Users can explore and deploy thousands of AI-powered applications across varied domains directly on the Hugging Face platform.
- "Spaces" enable easy hosting and sharing of ML demos and apps, fostering seamless community engagement and experimentation.

### Open Source & Collaboration  
- Hugging Face provides an open-source stack, allowing anyone to build powerful AI models and applications.
- Collaborative tools help build users’ portfolios and contribute to the wider ML community.

---

## Company Culture

Hugging Face champions an open, collaborative, and inclusive community spirit. The company thrives on transparency and shared learning, where contributors from diverse backgrounds unite to push the boundaries of AI.

- **Community-first mindset:** Collaborate with a global network passionate about AI advancements.
- **Innovation-driven:** Stay at the cutting edge with continuous release of new models and datasets.
- **Open source advocacy:** Accessibility and openness are core to the company’s values.
- **Supportive environment:** Enables sharing knowledge and building reputations within the AI ecosystem.

---

## Who Uses Hugging Face?

- **ML Researchers & Scientists:** Access to state-of-the-art open models and datasets for experimentation and publication.
- **Developers & Engineers:** Tools and infrastructure for production-grade AI apps suitable for enterprise and startups.
- **Enterprises & Startups:** Scalable AI deployments with enterprise-grade features and support.
- **Educators & Students:** Educational resources, models, and datasets to facilitate learning.
- **Data Scientists & Analysts:** Diverse datasets and NLP tools for analytic workflows.

---

## Careers & Opportunities

Hugging Face is growing rapidly and invites brilliant minds who are passionate about AI and open source to join their journey.

- Roles are available in **engineering, research, data science, community management, and product development**.
- Work alongside world-class AI experts and contribute to tools that impact millions.
- Emphasis on **remote-friendly** work culture with opportunities to engage globally.
- Continuous learning and professional growth are integral.

---

## Join the Future of AI

Whether you’re building the next big ML model, training on diverse datasets, or deploying AI applications, Hugging Face offers the tools, community, and platform to accelerate your AI journey.

Visit [huggingface.co](https://huggingface.co) to join the community, explore models and datasets, and start building today!

---

*Hugging Face — Where the Machine Learning Community Builds Tomorrow.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [39]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
          stream=True
    )    
    # response = ""
    # display_handle = display(Markdown(""), display_id=True)
    # for chunk in stream:
    #     response += chunk.choices[0].delta.content or ''
    #     update_display(Markdown(response), display_id=display_handle.display_id)
    # 定义响应内容
    response = ""
    # 渲染目标
    display_handle = display(Markdown(""),display_id=True)
    # 遍历并拿到增量然后拼接到响应内容
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
    # 更新显示
        update_display(Markdown(response),display_id=display_handle.display_id)


In [41]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5.4-nano
Found 18 relevant links


# Hugging Face Brochure  

## About Hugging Face  
Hugging Face is a pioneering AI company dedicated to building an open, collaborative, and ethical future for machine learning. It serves as the central platform for the global machine learning community to share, discover, and collaboratively develop open-source models, datasets, and applications. With over 2 million models and 500,000 datasets hosted, Hugging Face empowers ML engineers, scientists, and enthusiasts to innovate faster and together.  

## What We Offer  
- **Model Hub:** Access and contribute to over 2 million machine learning models cutting across modalities including text, image, video, audio, and even 3D.  
- **Datasets:** Explore and share from 500k+ curated datasets supporting diverse ML applications.  
- **Spaces:** Host and run ML apps and demos easily to showcase capabilities or share research.  
- **Open Source Stack:** Benefit from an open-source ecosystem fostering transparency, collaboration, and accelerated innovation.  
- **Enterprise Solutions:** Tailored plans to help organizations integrate, customize, and scale AI models securely.  

## Our Community & Customers  
Hugging Face is at the epicenter of the AI revolution, driven by a fast-growing global community — spanning independent researchers, startups, academia, and enterprise teams. Our platform is trusted by AI builders to share advanced tools and resources, facilitating breakthrough applications and research across industries.  

## Company Culture  
At Hugging Face, collaboration and openness are core values. We believe in democratizing access to AI, prioritizing ethical development, and fostering an inclusive environment where diversity and talent thrive. Our science team explores cutting-edge research, while the entire community participates in shaping the future of machine learning together.  

## Careers  
We are looking for passionate machine learning engineers, researchers, software developers, and community contributors eager to shape AI’s future. Join us to work in a dynamic, innovative environment that encourages learning, growth, and impact. Check our Careers page for open roles and start building tomorrow’s AI with us.  

## Brand and Visual Identity  
- Signature Colors: Bright Yellow (#FFD21E), Orange (#FF9D00), and Slate Gray (#6B7280)  
- Logos available in multiple formats (.svg, .png, .ai) to support community and enterprise branding needs.  

## Connect With Hugging Face  
- Website: [huggingface.co](https://huggingface.co)  
- GitHub, Twitter, LinkedIn, Discord: Active social and developer engagement channels.  
- Documentation & Blog: Extensive resources to learn, build, and collaborate.  

**Join the AI community building the future — explore, share, and innovate with Hugging Face today!**

In [38]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5.4-nano
Found 18 relevant links


# Hugging Face Brochure

## About Hugging Face

Hugging Face is the AI community and collaboration platform dedicated to building the future of machine learning (ML). It serves as a central hub where ML engineers, scientists, and end users can share, explore, discover, and experiment with open-source ML models, datasets, and applications. Hugging Face empowers the next generation of AI practitioners by fostering an open and ethical AI future through community-driven innovation.

## What We Offer

- **Models**: Access to over 2 million machine learning models covering various modalities including text, images, video, audio, and even 3D. These models are continuously updated by contributors worldwide.
- **Datasets**: Browse and use from over 500,000 datasets curated for different AI tasks to build and train models.
- **Spaces**: Host and deploy interactive ML applications with ease, enabling users to showcase projects or create new AI tools.
- **Buckets**: Secure storage solutions within the ecosystem for scalable AI workflows.
- **Open Source Stack**: A comprehensive open source software ecosystem that supports accelerated ML development and experimentation.

## Community & Innovation

Hugging Face is home to a fast-growing, vibrant community of machine learning enthusiasts and experts. The platform supports collaborative contributions and fosters learning, enabling community members to build their professional ML portfolios and share their work openly with the world.

The Hugging Face science team is at the cutting edge of research, exploring new techniques and technologies in NLP, computer vision, reinforcement learning, ethical AI, and more. The platform also offers extensive documentation, forums, and a community blog where users discuss the latest in AI research and applications.

## Company Culture

Hugging Face embraces a culture of openness, collaboration, and ethical responsibility in AI development. It values innovation grounded in community participation and transparency. By creating a welcoming environment that encourages sharing and experimentation, Hugging Face empowers contributors and users alike to push the boundaries of technology together.

## Customers & Use Cases

Hugging Face is trusted by a diverse range of users, from independent researchers and developers to enterprises incorporating AI into their workflows. Its tools and resources are ideal for:

- Building and deploying state-of-the-art natural language processing (NLP) models
- Developing computer vision applications and multimodal AI systems
- Applying reinforcement learning in various domains including robotics and gaming
- Creating audio and video analysis tools
- Driving innovation in healthcare, cybersecurity, and many other industries

## Careers at Hugging Face

Join a pioneering team at the heart of the AI revolution. Hugging Face offers exciting opportunities for talented individuals passionate about machine learning, software engineering, research science, and open source community building.

Careers at Hugging Face provide:

- The chance to work with cutting-edge AI technologies and open source projects
- A collaborative environment that encourages creativity, learning, and growth
- Engagement with a global, diverse AI community
- Opportunities to contribute to ethical and responsible AI development
- Flexible work culture, supporting innovation and work-life balance

Explore current openings and join Hugging Face to help build the future of AI.

---

For more information, visit [huggingface.co](https://huggingface.co)  
Connect with the community on GitHub, Twitter, LinkedIn, and Discord.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>